In [7]:
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import re
import string

# Принудительно устанавливаем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")


# Функция для мониторинга GPU
def print_gpu_usage():
    if torch.cuda.is_available():
        print(f"Использовано памяти: {torch.cuda.memory_allocated(0) / 1024 ** 2:.2f} MB")
        print(f"Зарезервировано памяти: {torch.cuda.memory_reserved(0) / 1024 ** 2:.2f} MB")


time_start = time.time()

# Загрузка данных
df = pd.read_csv('combined_data.csv')
df = df[df.text.notnull()]
texts = df.text.tolist()[:1000]  # Ограничим данные для теста
labels = df.label.tolist()[:1000]

print(f"Загружено {len(texts)} текстов")


# Упрощенный текстовый процессор
class TextProcessor:
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.word_to_idx = {}

    def build_vocab(self, texts):
        word_freq = {}
        for text in texts:
            words = self._tokenize(text)
            for word in words:
                word_freq[word] = word_freq.get(word, 0) + 1

        sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
        if self.max_features:
            sorted_words = sorted_words[:self.max_features]

        self.word_to_idx = {word: idx for idx, (word, _) in enumerate(sorted_words)}
        self.vocab_size = len(self.word_to_idx)
        print(f"Размер словаря: {self.vocab_size}")

    def _tokenize(self, text):
        text = str(text).lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return [word for word in text.split() if len(word) > 2]  # Слова от 3 символов

    def text_to_tensor(self, text):
        words = self._tokenize(text)
        vector = torch.zeros(self.vocab_size, dtype=torch.float32)
        for word in words:
            if word in self.word_to_idx:
                vector[self.word_to_idx[word]] = 1.0
        return vector


# Dataset
class TextDataset(Dataset):
    def __init__(self, texts, labels, processor):
        self.texts = texts
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        vector = self.processor.text_to_tensor(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return vector, label


# Инициализация
processor = TextProcessor(max_features=1000)
processor.build_vocab(texts)

# Dataset и DataLoader
dataset = TextDataset(texts, labels, processor)
train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Более сложная модель для создания нагрузки
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


# Создаем модель и СРАЗУ перемещаем на GPU
model = NeuralNetwork(processor.vocab_size).to(device)
print("Модель создана и перемещена на GPU")
print_gpu_usage()

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\nНачинаем обучение...")

# Обучение с мониторингом GPU
for epoch in range(100):
    model.train()
    total_loss = 0

    for i, (vectors, labels) in enumerate(train_loader):
        # ЯВНО перемещаем данные на GPU
        vectors = vectors.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Прямой проход
        outputs = model(vectors).squeeze()
        loss = criterion(outputs, labels)

        # Обратный проход
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Выводим использование GPU каждые 10 батчей
        if i % 10 == 0:
            print(f"Батч {i}, Loss: {loss.item():.4f}")
            print_gpu_usage()
            print("-" * 30)

    print(f'Epoch {epoch + 1}, Средний Loss: {total_loss / len(train_loader):.4f}')

# Тестирование
model.eval()
correct = total = 0
with torch.no_grad():
    for vectors, labels in test_loader:
        vectors, labels = vectors.to(device), labels.to(device)
        outputs = model(vectors).squeeze()
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {correct / total:.4f}")
print(f"Общее время: {time.time() - time_start:.2f} сек")

Используется устройство: cuda
Загружено 1000 текстов
Размер словаря: 1000
Модель создана и перемещена на GPU
Использовано памяти: 30.21 MB
Зарезервировано памяти: 46.00 MB

Начинаем обучение...
Батч 0, Loss: 0.7001
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Батч 10, Loss: 0.6237
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Батч 20, Loss: 0.3738
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Epoch 1, Средний Loss: 0.5682
Батч 0, Loss: 0.2343
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Батч 10, Loss: 0.3744
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Батч 20, Loss: 0.1359
Использовано памяти: 27.70 MB
Зарезервировано памяти: 46.00 MB
------------------------------
Epoch 2, Средний Loss: 0.1533
Батч 0, Loss: 0.0459
Использовано памяти: 27.70 MB
З

In [8]:
torch.save(model, 'spam_filter.pt')